# Assignment 2

## Instructions - Please read these instructions THOROUGHLY First! 

This is an individual homework assignment. The implication of this is that:

- You may discuss the problems in this assignment with other students in this course and your instructor/TA, but YOUR WORK MUST BE YOUR OWN.
- Do not show other students code or your own work on this assignment.
- You may consult external references, but not actively receive help from individuals not involved in this course.
- Cite all references outside of the course you used, including conversations with other students which were helpful. (This helps us give credit where it is due!). All references must use a commonly accepted reference format, for example, APA or IEEE (or another citation style of your choice).

If any of these rules seem ambiguous, please check with with your instructor for help interpreting them.

We suggest completing this assignment using the provided notebook. Each question should be answered using a SQL query (or combination or SQL queries) unless the text indicates that you may do something else. You may submit your queries embedded in Python, using SQLAlchemy or the MySQL Connector, or as plain text in Markdown.

## When you submit your work

Your submission will be graded manually. To ensure that everything goes smoothly, please follow these instructions to prepare your notebook for submission to the D2L Dropbox for Assignment 2:

- Please remove any print statments used to test your work (this is done by commenting them out)
- Please provide your solutions where asked; And please do not alter any other parts of this notebook.
- If you need to add cells to test your code please move them to the end of the notebook before submission- or you may include your commented out answers and tests in the cells provided

## Introduction

 In this assignment, we will focus familiarizing you with using SQL for data exploration, and continuing to cultivate a sense of curiosity about the datasets you encounter. We will be using a CSV File generated by the <b>City of Edmonton</b> containing Licensed Pets in the city. It is assumed that this table has been pre-cleaned (although potentially not entirely) so that you can work on the actual assignment more quickly. This assignment has four (4) parts: PARTS A, B, C, and D.
 
 To begin, start by importing the provided CSV into your own SQL database using SQLAlchemy, by filling in the lines below:

In [3]:
import pandas as pd
import sqlalchemy as sq

# read in your CSV as a dataframe
df = pd.read_csv("Licensed_Pets_by_Breed_and_Forward_Sortation_Area__FSA_.csv")

# connect to your database; include a cell at the bottom of this notebook to dispose of your engine object
engine = sq.create_engine("sqlite:///pets.db")  

# write your dataframe into a table
df.to_sql("pets", engine, if_exists="replace", index=False)


# demonstrate that your import has been successful by reading your database table as a dataframe, 
df2 = pd.read_sql("SELECT * FROM pets LIMIT 10", engine)

# and print some information (not the entire table) about your second dataframe
print(df2.info())
print(df2.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   FORWARD SORTATION AREA  10 non-null     object
 1   ANIMAL TYPE             10 non-null     object
 2   BREED                   10 non-null     object
 3   COUNT                   10 non-null     int64 
dtypes: int64(1), object(3)
memory usage: 452.0+ bytes
None
  FORWARD SORTATION AREA ANIMAL TYPE                BREED  COUNT
0                    T3L         Cat  DOMESTIC SHORT HAIR      1
1                    T5A         Cat  DOMESTIC SHORT HAIR    987
2                    T5A         Dog  RHODESIAN RIDGEBACK      3
3                    T5A         Dog       SIBERIAN HUSKY     50
4                    T5A         Dog              MALTESE     57


## Part A: Warm-up Questions (12 marks)

Answer the questions below, including the queries you used where necessary. Not all questions will require writing a SQL query to answer.

**(A1): (1 mark)**

How many records are there in total? (<b>Please show the SQL query</b>)

In [7]:
pd.read_sql("SELECT COUNT(*) AS total_records FROM pets;", engine)

,total_records
0,5626


**(A2): (1.5 mark)**

How many known areas were covered in the dataset? (<b>Please show the SQL query</b>)

In [15]:
pd.read_sql("""
SELECT COUNT(DISTINCT "FORWARD SORTATION AREA") AS known_areas
FROM pets
WHERE "FORWARD SORTATION AREA" IS NOT NULL
AND "FORWARD SORTATION AREA" <> '';
""", engine)

,known_areas
0,39


**(A3): (1.5 mark)**

How many unknown areas were covered in the dataset? (<b>Please show the SQL query</b>)

In [16]:
pd.read_sql("""
SELECT COUNT(*) AS unknown_area_rows
FROM pets
WHERE "FORWARD SORTATION AREA" IS NULL
   OR "FORWARD SORTATION AREA" = '';
""", engine)

,unknown_area_rows
0,4


**(A4): (4 marks)**

Explain what each of the following columns is used for. You may use the original page to guide your explanation (but you should cite it)
* FORWARD SORTATION AREA (1 mark)
* ANIMAL (1 mark)
* BREED (1 mark)
* COUNT (1 mark)


In [17]:
pd.read_sql("""
SELECT "FORWARD SORTATION AREA", SUM(COUNT) AS dog_count
FROM pets
WHERE "ANIMAL TYPE" = 'Dog'
GROUP BY "FORWARD SORTATION AREA"
ORDER BY dog_count DESC
LIMIT 5;
""", engine)

,FORWARD SORTATION AREA,dog_count
0,T5T,4537
1,T6L,4280
2,T5Y,3197
3,T5X,3030
4,T6R,2987


**(A5): (4 marks)**

Thinking outside the box, would it be possible to find the actual names of the community/munipalities/streets in this dataset? Why or why not? (<b>Please explain clearly and concisely.

No, we are not able to determine the specific names of community/munipalities/streets in this dataset. The dataset only provides the Forward Sortation Area (FSA), or the first 3 characters of the postal code. Each FSA covers a large area containing many streets and many households, and does not identify specific locations. For example, the top FSA for dogs is T5T with 4,537 licenses, but that only tells us a general region, not which streets the pets live on. Similarly, there are 39 known FSAs and 4 unknown ones, but none of this lets us pinpoint exact addresses. So, the dataset lets us see patterns at the area level, but not the precise locations of pet owners, which also helps protect their privacy.

## Part B: Simple questions (10 marks) 

For these queries, run a query which provides the answer.

**(B1): (2 marks)**

List in <b>descending order</b> Five (5) Areas with the highest numbers of licensed Dogs

In [19]:
pd.read_sql("""
SELECT "FORWARD SORTATION AREA", SUM(COUNT) AS dog_count
FROM pets
WHERE "ANIMAL TYPE" = 'Dog'
GROUP BY "FORWARD SORTATION AREA"
ORDER BY dog_count DESC
LIMIT 5;
""", engine)

,FORWARD SORTATION AREA,dog_count
0,T5T,4537
1,T6L,4280
2,T5Y,3197
3,T5X,3030
4,T6R,2987


**(B2): (2 marks)**

How many Dog Breeds were captured in this dataset?

In [20]:
pd.read_sql("""
SELECT COUNT(DISTINCT BREED) AS number_of_dog_breeds
FROM pets
WHERE "ANIMAL TYPE" = 'Dog';
""", engine)

,number_of_dog_breeds
0,247


**(B3): (2 marks)**

Which Cat Breed is licensed the most? (1 Mark) How many are there in total? (1 Mark)

In [21]:
pd.read_sql("""
SELECT BREED, SUM(COUNT) AS total
FROM pets
WHERE "ANIMAL TYPE" = 'Cat'
GROUP BY BREED
ORDER BY total DESC
LIMIT 1;
""", engine)

,BREED,total
0,DOMESTIC SHORT HAIR,20891


**(B4): (2 marks)**

Produce a table comprising of Areas and corresponding Counts for Cats of Breed types 'Himalayan' and 'Domestic Short Hair'

In [22]:
pd.read_sql("""
SELECT "FORWARD SORTATION AREA", BREED, SUM(COUNT) AS total
FROM pets
WHERE "ANIMAL TYPE" = 'Cat'
  AND BREED IN ('Himalayan', 'DOMESTIC SHORT HAIR')
GROUP BY "FORWARD SORTATION AREA", BREED
ORDER BY "FORWARD SORTATION AREA", BREED;
""", engine)

,FORWARD SORTATION AREA,BREED,total
0,None,DOMESTIC SHORT HAIR,1
1,T3L,DOMESTIC SHORT HAIR,1
2,T5A,DOMESTIC SHORT HAIR,987
3,T5B,DOMESTIC SHORT HAIR,600
4,T5C,DOMESTIC SHORT HAIR,470
5,T5E,DOMESTIC SHORT HAIR,863
6,T5G,DOMESTIC SHORT HAIR,428
7,T5H,DOMESTIC SHORT HAIR,523
8,T5J,DOMESTIC SHORT HAIR,60
9,T5K,DOMESTIC SHORT HAIR,467


**(B5): (2 marks)**

What is the ratio of Cats to Dogs in this dataset?

In [23]:
pd.read_sql("""
WITH cat_count AS (
    SELECT SUM(COUNT) AS cats
    FROM pets
    WHERE "ANIMAL TYPE" = 'Cat'
),
dog_count AS (
    SELECT SUM(COUNT) AS dogs
    FROM pets
    WHERE "ANIMAL TYPE" = 'Dog'
)
SELECT cats * 1.0 / dogs AS cat_to_dog_ratio
FROM cat_count, dog_count;
""", engine)

,cat_to_dog_ratio
0,0.541391


## Part C: Detailed analysis (20 marks)

Now consider being given a task to make sense of the distribution of licensed pets based on this dataset.

**(C1):(4 marks)**

Create two guiding questions to use in your analysis, and include them below as Markdown. As a starting point (and remember you are not limited to only these!), you may want to consider the following ideas:
- Focus on a specific set of data from the dataset that interest you
- Consider the areas with multiple breeds within the broad pet categories
- What would you say in terms of patterns of licensed pets within the specific areas of focus

Question 1) Which FSAs have the most diverse range of pet breeds, and does this differ between cats and dogs?

This will show the different municipalities with diverse breed liking pet owners and the extent to which a few breeds are monopolizing the market in certain municipalities.


Question 2) Are there FSAs where one type of pet (cats or dogs) is much more common than the other, and what might that tell us about local preferences?

This looks into the possibility of some areas being biased toward dogs or cats and it is also a means of identifying the patterns of pet ownership in the city.

**(C2): (12 marks)** 

Write at least four queries (that is, two queries for each question) which you believe will address one of your guiding questions. Clearly indicate which queries address your questions. You may wish to include a comment to explain why this query will help address your question.

Question 1: Breed Diversity by Area

Dog breeds per FSA

In [24]:
pd.read_sql("""
SELECT "FORWARD SORTATION AREA", COUNT(DISTINCT BREED) AS dog_breed_diversity
FROM pets
WHERE "ANIMAL TYPE" = 'Dog'
GROUP BY "FORWARD SORTATION AREA"
ORDER BY dog_breed_diversity DESC;
""", engine)

,FORWARD SORTATION AREA,dog_breed_diversity
0,T6L,170
1,T5T,168
2,T6J,167
3,T5R,158
4,T6H,150
5,T6R,149
6,T5Y,148
7,T6C,147
8,T5X,147
9,T6E,144


Cat breeds per FSA

In [25]:
pd.read_sql("""
SELECT "FORWARD SORTATION AREA", COUNT(DISTINCT BREED) AS cat_breed_diversity
FROM pets
WHERE "ANIMAL TYPE" = 'Cat'
GROUP BY "FORWARD SORTATION AREA"
ORDER BY cat_breed_diversity DESC;
""", engine)


,FORWARD SORTATION AREA,cat_breed_diversity
0,T6L,43
1,T6J,40
2,T5X,40
3,T5T,39
4,T6E,36
5,T6C,35
6,T6R,33
7,T6K,33
8,T5R,33
9,T5B,33


As we can see in the results, certain FSAs contain a significantly larger diversity of dog and cat breeds than others, where T6L has the highest number of breeds (170 breeds), with T5T and T6J closely following at 168 and 167 breeds, respectively. For cats, T6L also has the largest breed count (43) followed by T6J and T5X with an equal level of breeds (40). The results indicate that some areas of Edmonton have more dog and cat owners with more diverse breed preferences while other areas have owners with less diverse breed usage.

Question 2: Pet Type Dominance by Area

Total cats and dogs per FSA

In [26]:
pd.read_sql("""
SELECT "FORWARD SORTATION AREA",
       SUM(CASE WHEN "ANIMAL TYPE"='Cat' THEN COUNT ELSE 0 END) AS total_cats,
       SUM(CASE WHEN "ANIMAL TYPE"='Dog' THEN COUNT ELSE 0 END) AS total_dogs
FROM pets
GROUP BY "FORWARD SORTATION AREA"
ORDER BY "FORWARD SORTATION AREA";
""", engine)

,FORWARD SORTATION AREA,total_cats,total_dogs
0,None,2,3
1,T3L,1,0
2,T5A,1612,2849
3,T5B,979,1492
4,T5C,724,1340
5,T5E,1370,2302
6,T5G,710,1101
7,T5H,876,928
8,T5J,101,111
9,T5K,742,559


Ratio of cats to dogs per FSA

In [27]:
pd.read_sql("""
SELECT "FORWARD SORTATION AREA",
       SUM(CASE WHEN "ANIMAL TYPE"='Cat' THEN COUNT ELSE 0 END) * 1.0 /
       NULLIF(SUM(CASE WHEN "ANIMAL TYPE"='Dog' THEN COUNT ELSE 0 END), 0) AS cat_to_dog_ratio
FROM pets
GROUP BY "FORWARD SORTATION AREA"
ORDER BY cat_to_dog_ratio DESC;
""", engine)

,FORWARD SORTATION AREA,cat_to_dog_ratio
0,T5K,1.327370
1,T5H,0.943966
2,T5J,0.909910
3,T6E,0.866071
4,T6G,0.837171
5,T5M,0.818596
6,T5V,0.675000
7,None,0.666667
8,T5B,0.656166
9,T6H,0.654898


By checking out the number of cats and dogs in different areas and comparing their ratios, we can figure out which places like cats more or dogs. An example is the T5K FSA, which has the highest ratio at 1.33; this means there are more cats than dogs. Other FSAs like T5Y (0.42 ratio) and T6R (0.34 ratio) are dog-dominated. Thus, this pattern shows that ownership preferences for pets vary geographically, favoring cats in some neighborhoods and dogs in others.

**(C3): (4 marks)**

What kind of data would be interesting to have to be able to make more sense of the dataset? (1 Mark) Discuss how you could use this additional information to extend one of your guiding questions. (3 Marks)

It would be beneficial if we could have more data, for example; the ages of pets, and the demographic features of owners such as the size of households, age and income for instance. Pet ages can help us learn about different places, like for instance the areas that are more likely to have younger or older pets and also the patterns of new pet ownership around those areas. Demographic features can give us a clearer picture of the reasons why certain areas might have preferences for certain breeds or types of pets. If we could obtain data on the neighbourhood level or even the full postal codes, that would allow for tighter analyses than just FSAs, revealing patterns at the community level versus the broader region level. With this additional information, we could also expand our guiding questions to not only look at where pets are concentrating but also answer the question of what drives popularity of types or breeds of pets allowing for deeper insights into the provisioning and diversity of licensed pets in Edmonton.

## Part D: Reflection (5 marks)

In 100 to 250 words, identify a concept you have found difficult or confusing from this assignment. Reflect on how your previous learning or experience helped you to understand this concept. Provide your reflection using markdown in the cell below

At first, I found it quite tough to grasp the concept of SQL aggregation using an actual dataset for this assignment, especially when GROUP BY was employed to count distinct breeds across multiple FSAs. Combining the two conditions of dogs and cats and summarizing them correctly was at first a confusing task. However, my prior experience with pivot tables in Excel helped me grasp the concept, as GROUP BY functions similarly by summarizing data into meaningful categories. Thus, running queries to summarize the total counts, the diversity of breeds, and the ratio of cats to dogs also revealed to me the necessity of dealing with missing values or unknown values since some FSAs had incomplete data. Moreover, this was also part of the interpretation piece because this was equally important, not just to see the numbers but also to be aware of the patterns and trends that the output pointed out. One such case was T6L which had the highest diversity for both cats and dogs, and thinking critically about what that meant for pet ownership. In general, the assignment was a confidence booster for me in the use of SQL as a tool in exploratory data analysis, and I came to know better the technique of merging numerical outputs with insightful interpretation to tell a clear story from the data.

Use this cell to include some code to dispose of your SQLAlchemy engine object

In [31]:
engine.dispose()

## References

City of Edmonton (2019). Retrieved From: https://data.edmonton.ca/Demographics/Licensed-Pets-by-Breed-and-Forward-Sortation-Area-/bqmh-j34s